# Counterfactual decision support for pre-trip EV charging — reproduction notebook

This notebook reproduces every result of the article from two public datasets:

- **Dataset-1** — `electric_vehicles_spec_2025.csv`, specifications of 478 electric vehicles (Kaggle: `urvishahir/electric-vehicle-specifications-dataset-2025`).
- **Dataset-2** — `station_data_dataverse.csv`, 3,395 workplace charging sessions (Kaggle: `mdasrafulislamraihat/station-data-dataverse`).

Place both files in `EV_DATA` (default `/content/ev/data`). Outputs go to `EV_OUT` (default `/content/ev/out`); small result files are copied to `MyDrive/Outputs/EV_DSS/` when Google Drive is mounted.

| Stage | Content | Article |
|---|---|---|
| 1 | Data, group-aware split, nested cross-validation model selection, held-out evaluation | Section 3.2, Tables 2, 13 |
| 2 | Vehicle efficiency, nearest-energy matching sensitivity, trip-level system evaluation, session adequacy, band stability | Sections 3.3, 3.5, 4.5; Tables 5, 6, 15 |
| 3 | DiCE counterfactual feasibility (vehicle selection, charging plan, independent session variables) | Section 3.4; Table 14 |
| 4 | Case studies (out-of-fold predictions), figures 3–15 | Section 4; Table 10 |
| 5 | Architecture diagram (Figure 1) | Section 3.1 |

In [ ]:
# !pip install -q xgboost==3.4.1 scikit-learn==1.9.1 dice-ml shap
import os, shutil, glob
os.environ.setdefault('EV_DATA', '/content/ev/data'); os.environ.setdefault('EV_OUT', '/content/ev/out')
os.makedirs(os.environ['EV_DATA'], exist_ok=True); os.makedirs(os.environ['EV_OUT'], exist_ok=True)
def backup():
    if os.path.isdir('/content/drive/MyDrive'):
        dst = '/content/drive/MyDrive/Outputs/EV_DSS'; os.makedirs(dst, exist_ok=True)
        for f in glob.glob(os.environ['EV_OUT'] + '/*.json') + glob.glob(os.environ['EV_OUT'] + '/*.csv') + glob.glob(os.environ['EV_OUT'] + '/figs/*.png'):
            shutil.copy(f, dst)
        print('results copied to', dst)
print('data files:', os.listdir(os.environ['EV_DATA']))

## Stage 1 — Range estimator: group-aware split and nested cross-validation
Vehicles with identical feature vectors form one group and never straddle a split. Four model families × three feature sets are tuned by inner group cross-validation and compared by an outer 5-fold × 3-repeat stratified group cross-validation; the criterion is the lowest mean outer-fold MAE. Each model prints its result as it finishes (about 15 minutes on two CPU cores).

In [ ]:
import numpy as np, pandas as pd, warnings, json, time, os
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, RepeatedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, median_absolute_error, max_error, explained_variance_score
SEED = 42
DATA = os.environ.get('EV_DATA', '/content/ev/data'); OUT = os.environ.get('EV_OUT', '/content/ev/out'); os.makedirs(OUT, exist_ok=True)
R = {}
# ---------------- Dataset-1 ----------------
raw = pd.read_csv(f'{DATA}/electric_vehicles_spec_2025.csv')
d = raw.dropna(subset=['range_km']).copy()
d['cargo_volume_l'] = pd.to_numeric(d['cargo_volume_l'], errors='coerce')
for col in d.select_dtypes(include=['number']).columns: d[col] = d[col].fillna(d[col].median())
for col in d.select_dtypes(include='object').columns: d[col] = d[col].fillna(d[col].mode().iloc[0])
d['car_body_type_int'] = LabelEncoder().fit_transform(d['car_body_type'])
d['drivetrain_int'] = LabelEncoder().fit_transform(d['drivetrain'])
print('records', len(d))
FEATS10 = ["battery_capacity_kWh","top_speed_kmh","fast_charging_power_kw_dc","torque_nm","width_mm","length_mm","towing_capacity_kg","number_of_cells","cargo_volume_l","car_body_type_int"]
STRICT = ["battery_capacity_kWh","top_speed_kmh","fast_charging_power_kw_dc","torque_nm","width_mm","length_mm","towing_capacity_kg","number_of_cells","height_mm","seats","drivetrain_int","car_body_type_int"]
y = d['range_km'].astype(float)
bins = pd.cut(y, bins=10, labels=False)
# vehicles with identical feature vectors (e.g. trim variants) form one group and never straddle a split
groups = d[FEATS10].astype(str).agg('|'.join, axis=1).factorize()[0]
R['groups'] = dict(n_rows=len(d), n_groups=int(groups.max() + 1), rows_in_shared_vectors=int(pd.Series(groups).duplicated(keep=False).sum()))
print(R['groups'])
a_, b_ = next(StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=SEED).split(d, bins, groups))
idx_tr, idx_te = d.index[a_], d.index[b_]
# plain stratified split that ignores groups, kept only to quantify the effect of grouping
u_tr, u_te = train_test_split(d.index, test_size=0.5, random_state=SEED, stratify=bins)
def feats(name):
    if name == 'paper10': return d[FEATS10].astype(float)
    if name == 'strict12': return d[STRICT].astype(float)
    if name == 'strict12_onehot':
        X = d[[c for c in STRICT if c != 'car_body_type_int']].astype(float)
        return pd.concat([X, pd.get_dummies(d['car_body_type'], prefix='body').astype(float)], axis=1)
CANDS = {
 'Ridge': (make_pipeline(StandardScaler(), Ridge()), {'ridge__alpha': [0.1, 1, 10, 100]}),
 'Random Forest': (RandomForestRegressor(random_state=SEED, n_jobs=-1), {'n_estimators': [300], 'max_features': [0.33, 0.66, 1.0], 'min_samples_leaf': [1, 3]}),
 'Gradient Boosting': (GradientBoostingRegressor(random_state=SEED), {'n_estimators': [200, 500], 'learning_rate': [0.05, 0.1], 'max_depth': [2, 3]}),
 'XGBoost': (XGBRegressor(random_state=SEED, n_jobs=2, verbosity=0), {'n_estimators': [300, 600], 'learning_rate': [0.05], 'max_depth': [3, 6], 'subsample': [0.8, 1.0]}),
}
# ---------------- nested CV model selection (criterion: mean outer-fold MAE) ----------------
def outer_splits(X):
    for rep in range(3):
        for a, b in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED + rep).split(X, bins, groups): yield a, b
rows = []
for fs in ['paper10', 'strict12', 'strict12_onehot']:
    X = feats(fs)
    for name, (est, grid) in CANDS.items():
        t0 = time.time(); maes = []; r2s = []
        for k, (a, b) in enumerate(outer_splits(X)):
            gs = GridSearchCV(est, grid, cv=GroupKFold(3), scoring='neg_mean_absolute_error', n_jobs=1)
            gs.fit(X.iloc[a], y.iloc[a], groups=groups[a]); p = gs.predict(X.iloc[b])
            maes.append(mean_absolute_error(y.iloc[b], p)); r2s.append(r2_score(y.iloc[b], p))
        rows.append(dict(features=fs, model=name, MAE_mean=np.mean(maes), MAE_sd=np.std(maes, ddof=1), MAE_se=np.std(maes, ddof=1) / np.sqrt(len(maes)),
                         R2_mean=np.mean(r2s), R2_sd=np.std(r2s, ddof=1), secs=time.time() - t0))
        print(f'{fs:16s} {name:18s} outer MAE {np.mean(maes):6.2f} ± {np.std(maes, ddof=1):5.2f}  R2 {np.mean(r2s):.4f}  ({time.time()-t0:.0f}s)', flush=True)
NC = pd.DataFrame(rows).sort_values('MAE_mean'); NC.to_csv(f'{OUT}/nested_cv.csv', index=False)
best = NC.iloc[0]
# one-standard-error rule: simplest configuration within 1 SE of the best is reported as an alternative
print('\nBEST by mean outer MAE:', best['features'], best['model'], round(best['MAE_mean'], 2))
R['nested_cv'] = NC.to_dict('records'); R['selected'] = dict(features=best['features'], model=best['model'])
# ---------------- final model: inner-CV tuning on the training half, evaluation on the held-out half ----------------
X = feats(best['features']); est, grid = CANDS[best['model']]
gs = GridSearchCV(est, grid, cv=GroupKFold(5), scoring='neg_mean_absolute_error').fit(X.loc[idx_tr], y.loc[idx_tr], groups=groups[d.index.get_indexer(idx_tr)])
model = gs.best_estimator_; R['selected']['params'] = {k: (v if not isinstance(v, np.generic) else v.item()) for k, v in gs.best_params_.items()}
p = model.predict(X.loc[idx_te]); yt = y.loc[idx_te]
M = dict(R2=r2_score(yt, p), MSE=mean_squared_error(yt, p), RMSE=mean_squared_error(yt, p) ** .5, MAE=mean_absolute_error(yt, p), MAPE=mean_absolute_percentage_error(yt, p),
         MedAE=median_absolute_error(yt, p), MaxErr=max_error(yt, p), EVS=explained_variance_score(yt, p))
perc = np.abs(p - yt.values) / yt.values * 100
M.update(low_lt10=float((perc < 10).mean() * 100), mod_10_20=float(((perc >= 10) & (perc < 20)).mean() * 100), high_ge20=float((perc >= 20).mean() * 100))
R['heldout'] = M; print('held-out', {k: round(v, 4) for k, v in M.items()}, 'n_train', len(idx_tr), 'n_test', len(idx_te))
# legacy configuration on the same split for reference
leg = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=SEED).fit(feats('paper10').loc[idx_tr], y.loc[idx_tr])
pl = leg.predict(feats('paper10').loc[idx_te]); R['legacy_xgb_heldout'] = dict(R2=r2_score(yt, pl), MAE=mean_absolute_error(yt, pl), RMSE=mean_squared_error(yt, pl) ** .5)
leg2 = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=SEED).fit(feats('paper10').loc[u_tr], y.loc[u_tr])
pu = leg2.predict(feats('paper10').loc[u_te]); R['legacy_xgb_ungrouped_split'] = dict(R2=r2_score(y.loc[u_te], pu), MAE=mean_absolute_error(y.loc[u_te], pu), RMSE=mean_squared_error(y.loc[u_te], pu) ** .5)
print('legacy XGB, previous ungrouped split', R['legacy_xgb_ungrouped_split'])
print('legacy XGB held-out', R['legacy_xgb_heldout'])
import pickle
pickle.dump(dict(model=model, features=best['features'], idx_tr=list(idx_tr), idx_te=list(idx_te), pred_te=p), open(f'{OUT}/model.pkl', 'wb'))
json.dump(R, open(f'{OUT}/p1.json', 'w'), indent=1, default=float)


## Stage 2 — Efficiency, dataset linkage and trip-level evaluation
Compares the fixed 6 km/kWh with catalogue-implied, rated and model-derived efficiencies; tests nearest-energy matching of vehicles to sessions; evaluates five formulations of the charge-before-departure decision on 12,000 simulated trips of held-out vehicles.

In [ ]:
import numpy as np, pandas as pd, warnings, json, os, pickle, time
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.model_selection import train_test_split
SEED = 42; rng = np.random.default_rng(SEED)
DATA = os.environ.get('EV_DATA', '/content/ev/data'); OUT = os.environ.get('EV_OUT', '/content/ev/out')
R = {}
# ---------------- data and selected model ----------------
raw = pd.read_csv(f'{DATA}/electric_vehicles_spec_2025.csv')
d = raw.dropna(subset=['range_km']).copy()
d['cargo_volume_l'] = pd.to_numeric(d['cargo_volume_l'], errors='coerce')
for col in d.select_dtypes(include=['number']).columns: d[col] = d[col].fillna(d[col].median())
for col in d.select_dtypes(include='object').columns: d[col] = d[col].fillna(d[col].mode().iloc[0])
d['car_body_type_int'] = LabelEncoder().fit_transform(d['car_body_type']); d['drivetrain_int'] = LabelEncoder().fit_transform(d['drivetrain'])
FEATS10 = ["battery_capacity_kWh","top_speed_kmh","fast_charging_power_kw_dc","torque_nm","width_mm","length_mm","towing_capacity_kg","number_of_cells","cargo_volume_l","car_body_type_int"]
STRICT = ["battery_capacity_kWh","top_speed_kmh","fast_charging_power_kw_dc","torque_nm","width_mm","length_mm","towing_capacity_kg","number_of_cells","height_mm","seats","drivetrain_int","car_body_type_int"]
def feats(name):
    if name == 'paper10': return d[FEATS10].astype(float)
    if name == 'strict12': return d[STRICT].astype(float)
    X = d[[c for c in STRICT if c != 'car_body_type_int']].astype(float)
    return pd.concat([X, pd.get_dummies(d['car_body_type'], prefix='body').astype(float)], axis=1)
P1 = pickle.load(open(f'{OUT}/model.pkl', 'rb')); model = P1['model']; X = feats(P1['features'])
idx_tr, idx_te = P1['idx_tr'], P1['idx_te']; y = d['range_km'].astype(float)
Rhat_all = pd.Series(model.predict(X), index=d.index)          # in-sample on train half, out-of-sample on test half
te = d.loc[idx_te].copy(); te['Rhat'] = Rhat_all.loc[idx_te].values
cap = te['battery_capacity_kWh'].values
# ---------------- 1. efficiency: the fixed 6 km/kWh against vehicle-specific values ----------------
te['eta_true'] = te['range_km'] / te['battery_capacity_kWh']            # km per kWh implied by catalogue range (truth used below)
te['eta_rated'] = 1000 / te['efficiency_wh_per_km']                     # rated consumption field
te['eta_pred'] = te['Rhat'] / te['battery_capacity_kWh']               # vehicle-specific efficiency from the range model
R['efficiency'] = dict(eta_true=te['eta_true'].describe().to_dict(), eta_rated=te['eta_rated'].describe().to_dict(),
                       eta_pred_MAPE=float(np.mean(np.abs(te.eta_pred - te.eta_true) / te.eta_true) * 100),
                       eta_fixed6_MAPE=float(np.mean(np.abs(6.0 - te.eta_true) / te.eta_true) * 100),
                       eta_rated_MAPE=float(np.mean(np.abs(te.eta_rated - te.eta_true) / te.eta_true) * 100),
                       frac_true_below_6=float((te.eta_true < 6).mean()))
print('efficiency', {k: (round(v, 3) if isinstance(v, float) else '') for k, v in R['efficiency'].items()}, flush=True)
# ---------------- 2. Dataset-2: what it is, and the original energy bridge under different efficiency assumptions ----------------
d2 = pd.read_csv(f'{DATA}/station_data_dataverse.csv'); d2 = d2[d2['chargeTimeHrs'] > 0].copy()
d2['P'] = d2['kwhTotal'] / d2['chargeTimeHrs']
R['dataset2'] = dict(n=len(d2), kwh_q=d2.kwhTotal.quantile([.5, .9, .99, 1]).to_dict(), P_q=d2.P.quantile([.5, .9, .99, 1]).to_dict(),
                     frac_P_le_7_7=float((d2.P <= 7.7).mean()), hours_median=float(d2.chargeTimeHrs.median()))
def bridge(Rkm, eta):
    E = Rkm / eta; i = (d2['kwhTotal'] - E).abs().idxmin(); s = d2.loc[i]
    return E, i, abs(s.kwhTotal - E), int(np.ceil(E / max(s.kwhTotal, 1e-9))), Rkm / (s.P * eta) if s.P > 0 else np.nan
sens = []
base_match = np.array([bridge(r, 6.0)[1] for r in te.Rhat])
for lab, etas in [('4', np.full(len(te), 4.)), ('5', np.full(len(te), 5.)), ('6 (fixed)', np.full(len(te), 6.)), ('7', np.full(len(te), 7.)), ('8', np.full(len(te), 8.)),
                  ('vehicle-specific (predicted)', te.eta_pred.values)]:
    out = [bridge(r, e) for r, e in zip(te.Rhat.values, etas)]
    E = np.array([o[0] for o in out]); match = np.array([o[1] for o in out]); gap = np.array([o[2] for o in out]); ns = np.array([o[3] for o in out])
    Etrue = te.range_km.values / te.eta_true.values  # = battery capacity
    sens.append(dict(eta=lab, E_median=float(np.median(E)), E_err_vs_true_MAPE=float(np.mean(np.abs(E - Etrue) / Etrue) * 100),
                     frac_matched_session_changed=float(np.mean(match != base_match)), gap_median=float(np.median(gap)),
                     frac_exceeds_single_session=float(np.mean(E > d2.kwhTotal.max())), sessions_needed_median=float(np.median(ns))))
    print('bridge', sens[-1], flush=True)
R['bridge_sensitivity'] = sens
# ---------------- 3. trip-level system evaluation (Monte Carlo over test vehicles x trips) ----------------
# Truth: energy per km implied by the catalogue range of each vehicle; usable energy = capacity; reserve 10 % of capacity.
REserve = 0.10; N_TRIPS = 50
P_L2 = 7.2  # kW, typical single-phase AC Level 2 wallbox (SAE J1772 Level 2, IEC 61851 mode 3)
lin = LinearRegression().fit(feats(P1['features']).loc[idx_tr], y.loc[idx_tr])
te['R_lin'] = lin.predict(feats(P1['features']).loc[idx_te])
te['R_phys'] = te['battery_capacity_kWh'] * te['eta_rated']
def evaluate(policy):
    acc = []; unsafe = []; over = []; Eerr = []; terr = []
    for k, (_, v) in enumerate(te.iterrows()):
        r2 = np.random.default_rng(1000 + k)
        dist = r2.uniform(20, 1.2 * v.range_km, N_TRIPS); soc = r2.uniform(0.2, 1.0, N_TRIPS)
        need_true = dist / v.eta_true + REserve * v.battery_capacity_kWh - soc * v.battery_capacity_kWh   # kWh to add before departure
        must_true = need_true > 0
        E_hat, must_hat = policy(v, dist, soc)
        acc.append(np.mean(must_hat == must_true)); unsafe.append(np.mean(~must_hat & must_true)); over.append(np.mean(must_hat & ~must_true))
        m = must_true & must_hat
        if m.any():
            Eerr.extend(np.abs(E_hat[m] - need_true[m])); terr.extend(np.abs(E_hat[m] - need_true[m]) / P_L2)
    return dict(decision_accuracy=float(np.mean(acc)), unsafe_rate=float(np.mean(unsafe)), unnecessary_charge_rate=float(np.mean(over)),
                energy_MAE_kWh=float(np.mean(Eerr)), time_MAE_h_at_L2=float(np.mean(terr)))
def pol_original(v, dist, soc):        # nominal full-charge range vs distance; SoC ignored; energy from fixed 6 km/kWh
    must = dist > v.Rhat
    return np.maximum(dist - v.Rhat, 0) / 6.0 + 0 * dist, must
def pol_fixed6_soc(v, dist, soc):       # SoC-aware, but fixed 6 km/kWh
    need = dist / 6.0 + REserve * v.battery_capacity_kWh - soc * v.battery_capacity_kWh; return need, need > 0
def pol_proposed(v, dist, soc):          # SoC-aware, vehicle-specific efficiency from the range model
    need = dist / v.eta_pred + REserve * v.battery_capacity_kWh - soc * v.battery_capacity_kWh; return need, need > 0
def pol_linear(v, dist, soc):
    e = v.R_lin / v.battery_capacity_kWh; need = dist / e + REserve * v.battery_capacity_kWh - soc * v.battery_capacity_kWh; return need, need > 0
def pol_physics(v, dist, soc):           # rated consumption field, no learning
    need = dist / v.eta_rated + REserve * v.battery_capacity_kWh - soc * v.battery_capacity_kWh; return need, need > 0
SYS = {}
for lab, pol in [('Original pipeline (nominal range, SoC ignored, 6 km/kWh)', pol_original), ('SoC-aware, fixed 6 km/kWh', pol_fixed6_soc),
                 ('Physics (rated consumption field)', pol_physics), ('SoC-aware, linear-regression range', pol_linear), ('Proposed (SoC-aware, model-derived efficiency)', pol_proposed)]:
    t0 = time.time(); SYS[lab] = evaluate(pol); print('system', lab, {k: round(v, 4) for k, v in SYS[lab].items()}, f'{time.time()-t0:.0f}s', flush=True)
R['system_eval'] = SYS
# ---------------- 4. workplace-session adequacy from Dataset-2 (distributional link, not a vehicle match) ----------------
def p_single_session(E): return float((d2.kwhTotal >= E).mean())
R['session_adequacy'] = {f'{E} kWh': p_single_session(E) for E in [5, 10, 15, 20, 25]}
print('session adequacy', R['session_adequacy'], flush=True)
# ---------------- 5. decision stability of the range bands under the empirical residual distribution ----------------
res = (te.range_km - te.Rhat).values
band = lambda r: np.digitize(r, [200, 400])
chg = []
for k in range(200):
    pert = te.Rhat.values + np.random.default_rng(k).choice(res, len(te))
    chg.append(np.mean(band(pert) != band(te.Rhat.values)))
near = np.min(np.abs(te.Rhat.values[:, None] - np.array([200, 400])[None, :]), axis=1)
R['band_stability'] = dict(change_rate_mean=float(np.mean(chg)), change_rate_sd=float(np.std(chg)),
                           frac_within_1MAE_of_boundary=float(np.mean(near < np.mean(np.abs(res)))),
                           band_accuracy_vs_true=float(np.mean(band(te.Rhat.values) == band(te.range_km.values))))
print('stability', R['band_stability'], flush=True)
te[['range_km', 'Rhat', 'battery_capacity_kWh', 'eta_true', 'eta_pred', 'eta_rated']].to_csv(f'{OUT}/test_vehicles.csv')
json.dump(R, open(f'{OUT}/p2.json', 'w'), indent=1, default=float)

backup()

## Stage 3 — Physical feasibility of DiCE counterfactuals

In [ ]:
import numpy as np, pandas as pd, warnings, json, os, pickle
warnings.filterwarnings('ignore')
import dice_ml
from dice_ml import Dice
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
SEED = 42
DATA = os.environ.get('EV_DATA', '/content/ev/data'); OUT = os.environ.get('EV_OUT', '/content/ev/out')
R = {}
raw = pd.read_csv(f'{DATA}/electric_vehicles_spec_2025.csv')
d = raw.dropna(subset=['range_km']).copy()
d['cargo_volume_l'] = pd.to_numeric(d['cargo_volume_l'], errors='coerce')
for col in d.select_dtypes(include=['number']).columns: d[col] = d[col].fillna(d[col].median())
for col in d.select_dtypes(include='object').columns: d[col] = d[col].fillna(d[col].mode().iloc[0])
d['car_body_type_int'] = LabelEncoder().fit_transform(d['car_body_type']); d['drivetrain_int'] = LabelEncoder().fit_transform(d['drivetrain'])
FEATS10 = ["battery_capacity_kWh","top_speed_kmh","fast_charging_power_kw_dc","torque_nm","width_mm","length_mm","towing_capacity_kg","number_of_cells","cargo_volume_l","car_body_type_int"]
STRICT = ["battery_capacity_kWh","top_speed_kmh","fast_charging_power_kw_dc","torque_nm","width_mm","length_mm","towing_capacity_kg","number_of_cells","height_mm","seats","drivetrain_int","car_body_type_int"]
P1 = pickle.load(open(f'{OUT}/model.pkl', 'rb')); model = P1['model']
def feats(name):
    if name == 'paper10': return d[FEATS10].astype(float)
    if name == 'strict12': return d[STRICT].astype(float)
    X = d[[c for c in STRICT if c != 'car_body_type_int']].astype(float)
    return pd.concat([X, pd.get_dummies(d['car_body_type'], prefix='body').astype(float)], axis=1)
X = feats(P1['features']); cols = list(X.columns); y = d['range_km'].astype(float)
Xtr = X.loc[P1['idx_tr']]; Xte = X.loc[P1['idx_te']]
FIXED = [c for c in cols if c.startswith('body_') or c in ('car_body_type_int', 'seats', 'drivetrain_int')]   # category of vehicle kept
CONT = [c for c in cols if c not in FIXED]
# ---------- plausibility of a vehicle configuration ----------
A = Xtr[CONT].values; mu = A.mean(0); Ci = np.linalg.pinv(np.cov(A.T))
md = lambda V: np.sqrt(np.einsum('ij,jk,ik->i', V - mu, Ci, V - mu))
MD99 = float(np.quantile(md(A), .99))
kpc = d.loc[P1['idx_tr'], 'battery_capacity_kWh'] / d.loc[P1['idx_tr'], 'number_of_cells']
KPC = (float(kpc.quantile(.01)), float(kpc.quantile(.99)))
def plausible(C):
    ok = md(C[CONT].values.astype(float)) <= MD99
    r = C['battery_capacity_kWh'].astype(float) / C['number_of_cells'].astype(float).clip(lower=1)
    return ok & (r >= KPC[0]).values & (r <= KPC[1]).values
mad = (Xtr - Xtr.median()).abs().median().replace(0, 1)
class Wrap:
    def __init__(s, m): s.m = m
    def predict(s, Q): return s.m.predict(pd.DataFrame(Q, columns=cols).astype(float))
df_dice = Xtr.copy(); df_dice['range_km'] = y.loc[P1['idx_tr']].values
EXP = Dice(dice_ml.Data(dataframe=df_dice, continuous_features=cols, outcome_name='range_km'),
           dice_ml.Model(model=Wrap(model), backend='sklearn', model_type='regressor'), method='random')
rng = np.random.default_rng(SEED); qidx = rng.choice(len(Xte), 30, replace=False)
pr_all = {c: [float(Xtr[c].min()), float(Xtr[c].max())] for c in cols}
out = {}
for mode in ['unconstrained (all features vary)', 'constrained (vehicle category fixed, plausibility filter)']:
    val = []; feas = []; prox = []; spars = []; nearest = []; ncf = []
    for qi in qidx:
        q = Xte.iloc[[qi]]; base = float(model.predict(q)[0]); tgt = [base * 1.2, base * 1.4]
        fv = cols if mode.startswith('unconstrained') else CONT
        try:
            cf = EXP.generate_counterfactuals(q, total_CFs=10, desired_range=tgt, features_to_vary=fv, permitted_range={c: pr_all[c] for c in fv}, random_seed=SEED, verbose=False)
            C = cf.cf_examples_list[0].final_cfs_df[cols].astype(float)
        except Exception:
            C = pd.DataFrame(columns=cols)
        if len(C) == 0: val.append(0); feas.append(0); ncf.append(0); continue
        p = model.predict(C); v = (p >= tgt[0]) & (p <= tgt[1]); f = plausible(C)
        if not mode.startswith('unconstrained'): C, p, v, f = C[f], p[f], v[f], f[f]
        ncf.append(len(C)); val.append(float(v.mean()) if len(C) else 0); feas.append(float(f.mean()) if len(C) else 0)
        if len(C):
            D = (C.values - q.values) / mad.values
            prox.append(float(np.abs(D).sum(1).mean())); spars.append(float((np.abs(D) > 1e-9).sum(1).mean()))
            nn = np.abs((C[CONT].values[:, None, :] - Xtr[CONT].values[None, :, :]) / mad[CONT].values).sum(2).min(1)
            nearest.append(float(np.median(nn)))
    out[mode] = dict(queries=len(qidx), queries_with_cf=int(np.sum(np.array(ncf) > 0)), valid_and_plausible_cf_share=float(np.mean([x for x, n in zip(np.array(val) * np.array(feas), ncf) if n > 0])), cf_per_query=float(np.mean(ncf)), validity=float(np.mean(val)), physically_plausible=float(np.mean(feas)),
                     proximity=float(np.mean(prox)), sparsity=float(np.mean(spars)), median_L1_to_nearest_real_vehicle=float(np.median(nearest)))
    print('vehicle CF', mode, {k: round(v, 3) for k, v in out[mode].items()}, flush=True)
R['vehicle_cf'] = out; R['plausibility'] = dict(MD99=MD99, kWh_per_cell=KPC)
# ---------- charging-session counterfactuals ----------
d2 = pd.read_csv(f'{DATA}/station_data_dataverse.csv'); d2 = d2[d2['chargeTimeHrs'] > 0].copy(); d2['P'] = d2.kwhTotal / d2.chargeTimeHrs
d2['rate'] = 6 * d2['P']
Xd = d2[['kwhTotal', 'chargeTimeHrs', 'P']]; yd = d2['rate']
Xa, Xb, ya, yb = train_test_split(Xd, yd, test_size=0.4, random_state=SEED)
las = Lasso(alpha=0.1).fit(Xa, ya)
E2 = Dice(dice_ml.Data(dataframe=pd.concat([Xa, ya], axis=1), continuous_features=list(Xd.columns), outcome_name='rate'),
          dice_ml.Model(model=las, backend='sklearn', model_type='regressor'), method='random')
viol = []; allcf = 0
for i in range(30):
    q = Xb.iloc[[i]]; base = float(las.predict(q)[0])
    try:
        C = E2.generate_counterfactuals(q, total_CFs=10, desired_range=[base * 1.3, base * 1.6], random_seed=SEED, verbose=False).cf_examples_list[0].final_cfs_df
    except Exception: continue
    rel = np.abs(C['P'] - C['kwhTotal'] / C['chargeTimeHrs'].clip(lower=1e-6)) / C['P'].abs().clip(lower=1e-6)
    viol.extend(list(rel > 0.05)); allcf += len(C)
R['session_cf_unconstrained'] = dict(n_cf=allcf, identity_violation_rate=float(np.mean(viol)))
print('session CF unconstrained', R['session_cf_unconstrained'], flush=True)
# constrained: decision variables are charger power P (charger classes) and dwell time t; energy follows E = P t
class Plan:   # range added (km) for a given vehicle efficiency; E = P * t holds by construction
    def __init__(s, eta): s.eta = eta
    def predict(s, Q):
        Q = np.asarray(pd.DataFrame(Q).values, dtype=float); return Q[:, 0] * Q[:, 1] * s.eta
grid = pd.DataFrame({'P': np.repeat([1.9, 3.7, 7.2, 11.0], 50), 't': np.tile(np.linspace(0.1, 10, 50), 4)})
res_c = []
te = pd.read_csv(f'{OUT}/test_vehicles.csv', index_col=0).iloc[:30]
for _, v in te.iterrows():
    need_km = 0.5 * v.range_km                       # request: add half of the nominal range before departure
    g = grid.copy(); g['added'] = g.P * g.t * v.eta_pred
    E3 = Dice(dice_ml.Data(dataframe=g, continuous_features=['P', 't'], outcome_name='added'),
              dice_ml.Model(model=Plan(v.eta_pred), backend='sklearn', model_type='regressor'), method='random')
    q = pd.DataFrame({'P': [3.7], 't': [1.0]})
    try:
        C = E3.generate_counterfactuals(q, total_CFs=5, desired_range=[need_km, need_km * 1.2], permitted_range={'P': [1.4, 19.2], 't': [0.1, 12.0]}, random_seed=SEED, verbose=False).cf_examples_list[0].final_cfs_df
        added_true = C.P.values * C.t.values * v.eta_true
        res_c.append(dict(n=len(C), valid=float(np.mean((C.added >= need_km) & (C.added <= 1.2 * need_km))), meets_true_need=float(np.mean(added_true >= need_km)),
                          P_le_19_2=float(np.mean(C.P <= 19.2)), median_hours=float(np.median(C.t))))
    except Exception: res_c.append(dict(n=0, valid=0, meets_true_need=0, P_le_19_2=1, median_hours=np.nan))
RC = pd.DataFrame(res_c)
R['session_cf_constrained'] = dict(vehicles=len(RC), cf_per_vehicle=float(RC.n.mean()), validity=float(RC.valid.mean()), identity_violation_rate=0.0,
                                   within_SAE_AC_limit=float(RC.P_le_19_2.mean()), meets_need_under_true_efficiency=float(RC.meets_true_need.mean()), median_hours=float(RC.median_hours.median()))
print('session CF constrained', R['session_cf_constrained'], flush=True)
json.dump(R, open(f'{OUT}/p3.json', 'w'), indent=1, default=float)

backup()

## Stage 4 — Case studies and figures 3–15
Case-study ranges are predicted out of fold.

In [ ]:
import numpy as np, pandas as pd, warnings, json, os, pickle
warnings.filterwarnings('ignore')
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
plt.rcParams.update({'font.size': 10, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})
DATA = os.environ.get('EV_DATA', '/content/ev/data'); OUT = os.environ.get('EV_OUT', '/content/ev/out'); FIG = OUT + '/figs'; os.makedirs(FIG, exist_ok=True)
raw = pd.read_csv(f'{DATA}/electric_vehicles_spec_2025.csv')
d = raw.dropna(subset=['range_km']).copy()
d['cargo_volume_l'] = pd.to_numeric(d['cargo_volume_l'], errors='coerce')
for col in d.select_dtypes(include=['number']).columns: d[col] = d[col].fillna(d[col].median())
for col in d.select_dtypes(include='object').columns: d[col] = d[col].fillna(d[col].mode().iloc[0])
d['car_body_type_int'] = LabelEncoder().fit_transform(d['car_body_type']); d['drivetrain_int'] = LabelEncoder().fit_transform(d['drivetrain'])
STRICT = ["battery_capacity_kWh","top_speed_kmh","fast_charging_power_kw_dc","torque_nm","width_mm","length_mm","towing_capacity_kg","number_of_cells","height_mm","seats","drivetrain_int","car_body_type_int"]
P1 = pickle.load(open(f'{OUT}/model.pkl', 'rb')); model = P1['model']; assert P1['features'] == 'strict12'
X = d[STRICT].astype(float); y = d.range_km
te = d.loc[P1['idx_te']]; pte = P1['pred_te']
R = {}
# ---------------- case studies: the three vehicles of the previous case studies ----------------
cases = {'A': dict(battery_capacity_kWh=25.0, torque_nm=113.0, length_mm=3700.0, trip_km=150, soc0=0.8),
         'B': dict(battery_capacity_kWh=81.2, torque_nm=795.0, length_mm=5060.0, trip_km=300, soc0=0.8),
         'C': dict(battery_capacity_kWh=21.3, torque_nm=220.0, length_mm=3631.0, trip_km=100, soc0=0.8)}
RES = 0.10; P_L2 = 7.2
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from xgboost import XGBRegressor
FEATS10 = ["battery_capacity_kWh","top_speed_kmh","fast_charging_power_kw_dc","torque_nm","width_mm","length_mm","towing_capacity_kg","number_of_cells","cargo_volume_l","car_body_type_int"]
groups = d[FEATS10].astype(str).agg('|'.join, axis=1).factorize()[0]; bins = pd.cut(y, bins=10, labels=False)
oof = pd.Series(np.zeros(len(d)), index=d.index)
for a_, b_ in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42).split(X, bins, groups):
    mm = XGBRegressor(**model.get_params()).fit(X.iloc[a_], y.iloc[a_]); oof.iloc[b_] = mm.predict(X.iloc[b_])
CS = {}
for k, c in cases.items():
    m = d[(d.battery_capacity_kWh == c['battery_capacity_kWh']) & (d.torque_nm == c['torque_nm']) & (d.length_mm == c['length_mm'])]
    v = m.iloc[0]; Rhat = float(oof.loc[v.name]);   # out-of-fold prediction: the vehicle is never in its model's training data
    cap = v.battery_capacity_kWh
    eta = Rhat / cap; avail = c['soc0'] * cap; need = c['trip_km'] / eta + RES * cap
    Ech = max(0.0, need - avail); Ech_cap = min(Ech, (1 - c['soc0']) * cap)
    feasible_after_full = c['trip_km'] / eta + RES * cap <= cap
    head = (1 - c['soc0']) * cap; enroute = max(0.0, Ech - head); Ech_dep = min(Ech, head)
    eta_t = v.range_km / cap; Ech_true = max(0.0, c['trip_km'] / eta_t + RES * cap - avail)
    t_L2 = Ech_dep / P_L2; t_DC = Ech_dep / v.fast_charging_power_kw_dc
    band = 'Low' if Rhat < 200 else ('Moderate' if Rhat < 400 else 'Good')
    CS[k] = dict(vehicle=f'{v.brand} {v.model}', in_test=bool(v.name in P1['idx_te']), range_true=float(v.range_km), range_pred=Rhat, eta_pred=eta,
                 trip_km=c['trip_km'], soc0=c['soc0'], E_trip=c['trip_km'] / eta, E_avail=avail, E_charge=Ech, feasible_single_charge=bool(feasible_after_full),
                 t_L2_h=t_L2, t_DC_h=t_DC, E_departure=Ech_dep, E_enroute=enroute, E_charge_catalogue=Ech_true, fast_dc_kw=float(v.fast_charging_power_kw_dc), band=band, top_speed=float(v.top_speed_kmh), torque=float(v.torque_nm),
                 width=float(v.width_mm), length=float(v.length_mm), towing=float(v.towing_capacity_kg),
                 p_workplace_session_covers=float(np.nan))
d2 = pd.read_csv(f'{DATA}/station_data_dataverse.csv'); d2 = d2[d2.chargeTimeHrs > 0]
for k in CS: CS[k]['p_workplace_session_covers'] = float((d2.kwhTotal >= CS[k]['E_charge']).mean()) if CS[k]['E_charge'] > 0 else 1.0
for k, v in CS.items(): print(k, {a: (round(b, 3) if isinstance(b, float) else b) for a, b in v.items()})
R['cases'] = CS
# ---------------- figures 3-7: held-out behaviour of the selected model ----------------
yt = te.range_km.values
f, a = plt.subplots(figsize=(10, 3.2)); a.plot(yt[:100], color='#b03030', lw=1.5, label='Actual'); a.plot(pte[:100], '--', color='#222', lw=1.2, label='Predicted')
a.set_xlabel('Held-out vehicle'); a.set_ylabel('range_km'); a.legend(frameon=False); f.savefig(f'{FIG}/Fig3.png'); plt.close(f)
f, a = plt.subplots(figsize=(6, 3.4)); a.hist(yt - pte, bins=35, color='#3268bf', alpha=.85); a.set_xlabel('Residual, actual − predicted (km)'); a.set_ylabel('Count'); f.savefig(f'{FIG}/Fig4.png'); plt.close(f)
perc = np.abs(pte - yt) / yt * 100; cats = [(perc < 10).mean() * 100, ((perc >= 10) & (perc < 20)).mean() * 100, (perc >= 20).mean() * 100]
f, a = plt.subplots(figsize=(6, 3.4)); b = a.bar(['Low (<10 %)', 'Moderate (10–20 %)', 'High (≥20 %)'], cats, color=['#2f855a', '#c05621', '#b03030'])
for r_, v_ in zip(b, cats): a.text(r_.get_x() + r_.get_width() / 2, v_ + 1, f'{v_:.2f} %', ha='center')
a.set_ylabel('Share of held-out vehicles (%)'); a.set_ylim(0, 105); f.savefig(f'{FIG}/Fig5.png'); plt.close(f)
import shap
ex = shap.TreeExplainer(model); Xte_ = X.loc[P1['idx_te']]
for fig_no, pos in [(6, 0), (7, int(np.argsort(np.abs(pte - 350))[0]))]:
    sv = ex(Xte_.iloc[[pos]]); vals = sv.values[0]; order = np.argsort(-np.abs(vals))[:8]
    f, a = plt.subplots(figsize=(7, 3.6)); a.barh([Xte_.columns[i] for i in order][::-1], vals[order][::-1], color=['#c53030' if v > 0 else '#2b6cb0' for v in vals[order][::-1]])
    a.axvline(0, color='k', lw=.8); a.set_xlabel('SHAP contribution to predicted range (km)')
    a.set_title(f'Actual {yt[pos]:.0f} km, predicted {pte[pos]:.2f} km, base value {float(sv.base_values[0]):.1f} km', fontsize=9)
    f.savefig(f'{FIG}/Fig{fig_no}.png'); plt.close(f)
    R[f'shap_fig{fig_no}'] = dict(actual=float(yt[pos]), pred=float(pte[pos]), base=float(sv.base_values[0]),
                                  top=[(Xte_.columns[i], float(vals[i]), float(Xte_.iloc[pos, i])) for i in order[:4]])
    print('SHAP', fig_no, R[f'shap_fig{fig_no}'])
# ---------------- figures 8-12: energy-domain, bridge, system and counterfactual analyses ----------------
P2 = json.load(open(f'{OUT}/p2.json')); P3 = json.load(open(f'{OUT}/p3.json')); tv = pd.read_csv(f'{OUT}/test_vehicles.csv', index_col=0)
f, a = plt.subplots(figsize=(7, 3.6))
a.hist(tv.eta_true, bins=30, alpha=.7, color='#2b6cb0', label='Catalogue range / battery capacity')
a.hist(tv.eta_rated, bins=30, alpha=.5, color='#c05621', label='Rated consumption field')
a.axvline(6, color='k', ls='--', lw=1.2, label='Fixed 6 km/kWh'); a.set_xlabel('Vehicle efficiency (km/kWh)'); a.set_ylabel('Held-out vehicles'); a.legend(frameon=False, fontsize=8)
f.savefig(f'{FIG}/Fig8.png'); plt.close(f)
f, a = plt.subplots(figsize=(7, 3.6))
a.hist(d2.kwhTotal, bins=40, color='#718096', alpha=.8, label='Delivered energy per workplace session (Dataset-2)')
a.hist(tv.Rhat / tv.eta_pred, bins=30, color='#c53030', alpha=.6, label='Full-range recharge energy, held-out vehicles')
a.set_xlabel('Energy (kWh)'); a.set_ylabel('Count'); a.legend(frameon=False, fontsize=8); f.savefig(f'{FIG}/Fig9.png'); plt.close(f)
S = P2['system_eval']; labs = list(S.keys()); short = ['Previous pipeline', 'SoC-aware,\nfixed 6 km/kWh', 'Physics\n(rated field)', 'SoC-aware,\nlinear range', 'Proposed']
f, a = plt.subplots(1, 2, figsize=(10, 3.6))
a[0].bar(short, [S[l]['decision_accuracy'] * 100 for l in labs], color='#2b6cb0'); a[0].set_ylabel('Charge-before-departure decision accuracy (%)'); a[0].tick_params(axis='x', labelsize=7)
a[1].bar(short, [S[l]['unsafe_rate'] * 100 for l in labs], color='#c53030'); a[1].set_ylabel('Unsafe decisions (%)'); a[1].tick_params(axis='x', labelsize=7)
f.savefig(f'{FIG}/Fig10.png'); plt.close(f)
V = P3['vehicle_cf']; mk = list(V.keys())
f, a = plt.subplots(1, 2, figsize=(9, 3.4))
a[0].bar(['Unconstrained', 'Constrained'], [V[m]['proximity'] for m in mk], color=['#c53030', '#2f855a']); a[0].set_ylabel('Proximity (MAD-normalised L1)')
a[1].bar(['Unconstrained', 'Constrained'], [V[m]['median_L1_to_nearest_real_vehicle'] for m in mk], color=['#c53030', '#2f855a']); a[1].set_ylabel('Distance to nearest real vehicle')
f.savefig(f'{FIG}/Fig11.png'); plt.close(f)
res = tv.range_km - tv.Rhat
f, a = plt.subplots(figsize=(7, 3.6)); a.scatter(tv.Rhat, res, s=10, color='#2b6cb0')
for b_ in (200, 400): a.axvline(b_, color='k', ls='--', lw=1)
a.axhline(0, color='grey', lw=.8); a.set_xlabel('Predicted range (km)'); a.set_ylabel('Residual (km)')
f.savefig(f'{FIG}/Fig12.png'); plt.close(f)
# ---------------- figures 13-15: advisory dashboards of the case studies ----------------
def dashboard(k, c, fn):
    lines = [f"Case {k}: {c['vehicle']}", '',
             f"Predicted full-charge range: {c['range_pred']:.1f} km   (catalogue {c['range_true']:.0f} km)",
             f"Model-derived efficiency: {c['eta_pred']:.2f} km/kWh",
             f"Trip: {c['trip_km']} km, departure state of charge {int(c['soc0']*100)} %, reserve 10 %",
             f"Trip energy {c['E_trip']:.1f} kWh; available {c['E_avail']:.1f} kWh; charge before departure {c['E_charge']:.1f} kWh",
             (f"Charging plan: add {c['E_departure']:.1f} kWh before departure ({c['t_L2_h']:.2f} h at 7.2 kW AC Level 2, or {c['t_DC_h']:.2f} h at {c['fast_dc_kw']:.0f} kW DC peak)" if c['E_charge'] > 0 else 'Charging plan: no charge needed before departure'),
             (f"En-route energy still required: {c['E_enroute']:.1f} kWh (trip plus reserve exceeds one full charge)" if c['E_enroute'] > 0 else f"Check with catalogue range: energy to add {c['E_charge_catalogue']:.1f} kWh"),
             f"Share of workplace sessions delivering this energy: {c['p_workplace_session_covers']*100:.1f} %", '',
             f"Status: {c['band']} driving range" + ('' if c['feasible_single_charge'] else '; trip exceeds one full charge — en-route charging required')]
    f, a = plt.subplots(figsize=(8, 3.2)); a.axis('off')
    for i, t in enumerate(lines): a.text(0.01, 0.95 - i * 0.095, t, fontsize=9, family='monospace', weight='bold' if i == 0 else 'normal', transform=a.transAxes)
    f.savefig(fn); plt.close(f)
for k, n in [('A', 13), ('B', 14), ('C', 15)]: dashboard(k, CS[k], f'{FIG}/Fig{n}.png')
json.dump(R, open(f'{OUT}/p4.json', 'w'), indent=1, default=float)
print('figures written')

backup()

## Stage 5 — Figure 1 (architecture)

In [ ]:
import os
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
f,a=plt.subplots(figsize=(11,4.6)); a.axis('off'); a.set_xlim(0,11); a.set_ylim(0,4.6)
def box(x,y,w,h,t,c):
    a.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.05',fc=c,ec='#333',lw=1)); a.text(x+w/2,y+h/2,t,ha='center',va='center',fontsize=8.5,wrap=True)
def arr(x1,y1,x2,y2): a.annotate('',xy=(x2,y2),xytext=(x1,y1),arrowprops=dict(arrowstyle='->',lw=1.2))
box(0.1,3.2,2.1,1.1,'Vehicle specification\n(Dataset-1, 478 EVs)','#e2e8f0')
box(0.1,1.7,2.1,1.1,'Trip inputs\ndistance, departure SoC,\nreserve','#e2e8f0')
box(0.1,0.2,2.1,1.1,'Charging-session record\n(Dataset-2, 3,395\nworkplace sessions)','#e2e8f0')
box(2.8,3.2,2.3,1.1,'Learned range estimator\n(XGBoost, nested-CV\nselected)','#bee3f8')
box(5.7,2.45,2.3,1.1,'Energy requirement\nE = d / η̂ + reserve − SoC·C\nη̂ = R̂ / C','#c6f6d5')
box(2.8,1.7,2.3,1.1,'DiCE vehicle-selection\ncounterfactuals\n(envelope-constrained)','#fefcbf')
box(5.7,0.9,2.3,1.1,'Charging plan\nE = P·t, P by charger class\n(SAE J1772 / IEC 61851)','#c6f6d5')
box(2.8,0.2,2.3,1.1,'Empirical session\nadequacy P(E_session ≥ E)','#fefcbf')
box(8.6,1.6,2.3,1.9,'Rule-based advisory\nmodule\nStatus · Driver actions ·\nSystem suggestions','#fed7d7')
arr(2.2,3.75,2.8,3.75); arr(5.1,3.75,5.7,3.1); a.plot([2.2,2.5,2.5,5.45,5.45],[1.95,1.95,1.5,1.5,2.75],color='k',lw=1.2); arr(5.45,2.75,5.7,2.75); arr(3.95,3.2,3.95,2.8); arr(6.85,2.45,6.85,2.0)
arr(5.1,0.75,5.7,1.3); arr(2.2,0.75,2.8,0.75); arr(8.0,3.0,8.6,2.8); arr(8.0,1.45,8.6,2.1)
f.savefig(os.environ['EV_OUT'] + '/figs/Fig1.png',dpi=200,bbox_inches='tight')

backup()